In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(r"..\..\Data\Main\modelling_dataset.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"Columns: {list(df.columns)}")

Shape: (589886, 15)
Tickers: 523
Columns: ['date', 'sentiment', 'ticker', 'Close', 'High', 'Low', 'Open', 'Volume', 'log_return', 'volatility_20d', 'next_day_return', 'target', 'sent_x_vol', 'market_vol', 'volatile_market']


## Calculate Technical Indicators

RSI (Relative Strength Index) — Measures whether a stock has been going up or down recently, on a scale of 0 to 100. Above 70 means it's been rising a lot (possibly overbought), below 30 means it's been falling a lot (possibly oversold). Uses 14-day window, which is the standard.
MACD (Moving Average Convergence Divergence) — Compares a short-term trend (12-day average) to a longer-term trend (26-day average). When the short-term is above the long-term, momentum is positive. When below, momentum is negative.
Moving averages ratio — Instead of the raw moving average (which varies per stock price), we calculate the ratio of Close to its 5-day and 10-day moving averages. If the ratio is above 1, the stock is trading above its recent average (bullish). Below 1, it's trading below (bearish).
Bollinger Band position — Where the current price sits within its Bollinger Bands (20-day average ± 2 standard deviations). A value near 1 means the price is at the upper band, near 0 means it's at the lower band.
Volume change — Today's volume divided by the 20-day average volume. Above 1 means higher than usual trading activity, which often accompanies significant price moves.
5-day momentum — The return over the last 5 trading days. Captures short-term trend direction.

In [3]:
def add_technical_indicators(data):
    """Calculate technical indicators per ticker."""
    
    df = data.copy()
    
    # RSI (14-day)
    delta = df.groupby('ticker')['Close'].diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.groupby(df['ticker']).transform(lambda x: x.rolling(14).mean())
    avg_loss = loss.groupby(df['ticker']).transform(lambda x: x.rolling(14).mean())
    rs = avg_gain / avg_loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # MACD (12-day EMA - 26-day EMA)
    ema12 = df.groupby('ticker')['Close'].transform(lambda x: x.ewm(span=12).mean())
    ema26 = df.groupby('ticker')['Close'].transform(lambda x: x.ewm(span=26).mean())
    df['macd'] = ema12 - ema26
    
    # Moving average ratios (price relative to its MA)
    ma5 = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(5).mean())
    ma10 = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(10).mean())
    df['ma5_ratio'] = df['Close'] / ma5
    df['ma10_ratio'] = df['Close'] / ma10
    
    # Bollinger Band position (where price sits within the bands, 0 to 1)
    ma20 = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(20).mean())
    std20 = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(20).std())
    upper_band = ma20 + 2 * std20
    lower_band = ma20 - 2 * std20
    df['bb_position'] = (df['Close'] - lower_band) / (upper_band - lower_band)
    
    # Volume change (today's volume relative to 20-day average)
    avg_volume = df.groupby('ticker')['Volume'].transform(lambda x: x.rolling(20).mean())
    df['volume_ratio'] = df['Volume'] / avg_volume
    
    # 5-day momentum (return over last 5 days)
    df['momentum_5d'] = df.groupby('ticker')['Close'].transform(lambda x: x.pct_change(5))
    
    return df

df = add_technical_indicators(df)

new_features = ['rsi', 'macd', 'ma5_ratio', 'ma10_ratio', 'bb_position', 'volume_ratio', 'momentum_5d']
print("=== New Technical Indicators ===\n")
for feat in new_features:
    non_null = df[feat].notna().sum()
    print(f"  {feat:<15} non-null: {non_null:,} ({non_null/len(df)*100:.1f}%)  mean: {df[feat].mean():.4f}")

print(f"\nTotal columns: {len(df.columns)}")
print(f"Shape: {df.shape}")

=== New Technical Indicators ===

  rsi             non-null: 582,564 (98.8%)  mean: 52.2878
  macd            non-null: 589,886 (100.0%)  mean: 0.5240
  ma5_ratio       non-null: 587,794 (99.6%)  mean: 1.0008
  ma10_ratio      non-null: 585,179 (99.2%)  mean: 1.0020
  bb_position     non-null: 579,949 (98.3%)  mean: 0.5363
  volume_ratio    non-null: 579,949 (98.3%)  mean: 1.0099
  momentum_5d     non-null: 587,271 (99.6%)  mean: 0.0033

Total columns: 22
Shape: (589886, 22)


In [4]:
df.to_csv(r"..\..\Data\Main\modelling_dataset_with_technicals.csv", index=False)

print(f"Saved: modelling_dataset_with_technicals.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nOriginal features: sentiment, volatility_20d, sent_x_vol")
print(f"New technical features: {new_features}")

Saved: modelling_dataset_with_technicals.csv
Shape: (589886, 22)
Columns: ['date', 'sentiment', 'ticker', 'Close', 'High', 'Low', 'Open', 'Volume', 'log_return', 'volatility_20d', 'next_day_return', 'target', 'sent_x_vol', 'market_vol', 'volatile_market', 'rsi', 'macd', 'ma5_ratio', 'ma10_ratio', 'bb_position', 'volume_ratio', 'momentum_5d']

Original features: sentiment, volatility_20d, sent_x_vol
New technical features: ['rsi', 'macd', 'ma5_ratio', 'ma10_ratio', 'bb_position', 'volume_ratio', 'momentum_5d']
